# Лабораторная работа 1.5. QR-разложение и QR-алгоритм

**Задание.** Реализовать алгоритм QR-разложения матриц в виде программы. На его основе разработать программу,
реализующую QR-алгоритм решения полной проблемы собственных значений произвольных матриц, задавая в качестве
входных данных матрицу и точность вычислений. С использованием разработанного ПО найти собственные значения матрицы.

$$
A = \begin{pmatrix} -2 & 7 & -6 \\ -1 & 9 & -4 \\ -1 & 8 & -3 \end{pmatrix}
$$

## Теория

### QR-разложение методом Хаусхолдера

Любую квадратную матрицу можно представить как $A = QR$, где $Q$ — ортогональная, $R$ — верхняя треугольная.
Разложение строится последовательным применением $n-1$ матриц отражения (Хаусхолдера)

$$H = E - \frac{2}{v^T v}\, v v^T, \qquad H = H^T = H^{-1}.$$

На $j$-м шаге вектор $v$ выбирается так, чтобы обнулить поддиагональные элементы $j$-го столбца:

$$v_i = 0\ (i < j), \qquad v_j = a_{jj} + \operatorname{sign}(a_{jj}) \sqrt{\textstyle\sum_{i \ge j} a_{ij}^2}, \qquad v_i = a_{ij}\ (i > j).$$

Тогда $R = H_{n-1} \cdots H_1 A$ и $Q = H_1 H_2 \cdots H_{n-1}$.

### QR-алгоритм

$$A^{(0)} = A, \qquad A^{(k)} = Q^{(k)} R^{(k)}, \qquad A^{(k+1)} = R^{(k)} Q^{(k)} = Q^{(k)T} A^{(k)} Q^{(k)}.$$

Все $A^{(k)}$ подобны $A$, поэтому имеют те же собственные значения. Последовательность сходится
к блочно-верхнетреугольной матрице:
- **вещественному** собственному значению соответствует диагональный элемент $a_{jj}$,
  под которым столбец обнуляется;
- **паре комплексно-сопряжённых** значений соответствует блок $2\times2$ на диагонали. Его элементы сами
  не сходятся, но сходятся корни характеристического уравнения блока:
  $$(a_{jj} - \lambda)(a_{j+1,j+1} - \lambda) - a_{j,j+1} a_{j+1,j} = 0.$$

**Критерии окончания** (проверяются для каждого столбца $j$):
- вещественное $\lambda_j$: $\; \left(\sum_{i > j} \left(a_{ij}^{(k)}\right)^2\right)^{1/2} \le \varepsilon$;
- комплексная пара в блоке $(j, j+1)$: $\; |\lambda^{(k)} - \lambda^{(k-1)}| \le \varepsilon$.

Скорость сходимости $a_{ij}^{(k)} \to 0$ определяется отношением $|\lambda_i / \lambda_j|$
для упорядоченных по модулю собственных значений.

## Реализация

In [1]:
import cmath
import math


def matmul(A, B):
    return [[sum(A[i][k] * B[k][j] for k in range(len(B))) for j in range(len(B[0]))]
            for i in range(len(A))]


def transpose(A):
    return [list(row) for row in zip(*A)]


def identity(n):
    return [[float(i == j) for j in range(n)] for i in range(n)]


def show(M, name=None, prec=6):
    if name:
        print(f"{name} =")
    for row in M:
        print("  [" + " ".join(f"{(0.0 if abs(v) < 1e-12 else v):{prec + 6}.{prec}f}" for v in row) + " ]")
    print()


def householder(A, j):
    """Матрица Хаусхолдера, обнуляющая элементы столбца j ниже диагонали."""
    n = len(A)
    norm = math.sqrt(sum(A[i][j] ** 2 for i in range(j, n)))
    v = [0.0] * n
    v[j] = A[j][j] + math.copysign(norm, A[j][j])
    for i in range(j + 1, n):
        v[i] = A[i][j]
    vv = sum(x * x for x in v)
    H = identity(n)
    if vv == 0:  # столбец уже нулевой
        return H
    for p in range(n):
        for q in range(n):
            H[p][q] -= 2 * v[p] * v[q] / vv
    return H


def qr_decompose(A):
    """QR-разложение методом отражений Хаусхолдера: A = QR."""
    n = len(A)
    R = [list(map(float, row)) for row in A]
    Q = identity(n)
    for j in range(n - 1):
        H = householder(R, j)
        R = matmul(H, R)
        Q = matmul(Q, H)
    return Q, R


def block_eigenvalues(A, j):
    """Корни характеристического уравнения блока 2x2 в строках/столбцах j, j+1."""
    a, b = A[j][j], A[j][j + 1]
    c, d = A[j + 1][j], A[j + 1][j + 1]
    tr, det = a + d, a * d - b * c
    disc = cmath.sqrt(tr * tr - 4 * det)
    return (tr + disc) / 2, (tr - disc) / 2


def qr_eigenvalues(A, eps, max_iter=10_000):
    """QR-алгоритм. Возвращает (собственные значения, число итераций, итоговая A^(k), история)."""
    n = len(A)
    Ak = [list(map(float, row)) for row in A]
    prev_blocks = {}   # корни блоков 2x2 на предыдущей итерации
    history = []       # для каждой итерации: максимум норм поддиагональных частей столбцов

    for k in range(1, max_iter + 1):
        Q, R = qr_decompose(Ak)
        Ak = matmul(R, Q)

        sub = [math.sqrt(sum(Ak[i][j] ** 2 for i in range(j + 1, n))) for j in range(n)]
        history.append(sub)

        eig, j, done = [], 0, True
        blocks = {}
        while j < n:
            if sub[j] <= eps:                       # вещественное значение
                eig.append(Ak[j][j])
                j += 1
            elif j + 1 < n and math.sqrt(sum(Ak[i][j] ** 2 for i in range(j + 2, n))) <= eps:
                pair = block_eigenvalues(Ak, j)     # блок 2x2
                blocks[j] = pair
                old = prev_blocks.get(j)
                if pair[0].imag == 0:               # вещественные корни: ждём обнуления столбца
                    done = False
                elif old is None or max(abs(pair[0] - old[0]), abs(pair[1] - old[1])) > eps:
                    done = False
                eig.extend(pair)
                j += 2
            else:
                done = False
                break
        prev_blocks = blocks
        if done:
            return eig, k, Ak, history
    raise RuntimeError("QR-алгоритм не сошёлся за max_iter итераций")


def fmt(z):
    z = complex(z)
    if abs(z.imag) < 1e-12:
        return f"{z.real:12.6f}"
    return f"{z.real:12.6f} {'+' if z.imag >= 0 else '-'} {abs(z.imag):.6f}i"

## Исходные данные

In [2]:
A = [
    [-2, 7, -6],
    [-1, 9, -4],
    [-1, 8, -3],
]
eps = 1e-6

show(A, "A", prec=0)
print(f"eps = {eps}")

A =
  [    -2      7     -6 ]
  [    -1      9     -4 ]
  [    -1      8     -3 ]

eps = 1e-06


## QR-разложение матрицы A

In [3]:
Q, R = qr_decompose(A)
show(Q, "Q")
show(R, "R")
show(matmul(Q, R), "Проверка Q·R")
show(matmul(transpose(Q), Q), "Проверка Q^T·Q")

Q =
  [   -0.816497     0.573068    -0.070186 ]
  [   -0.408248    -0.659028    -0.631676 ]
  [   -0.408248    -0.487108     0.772049 ]

R =
  [    2.449490   -12.655697     7.756718 ]
  [    0.000000    -5.816643     0.659028 ]
  [    0.000000     0.000000     0.631676 ]

Проверка Q·R =
  [   -2.000000     7.000000    -6.000000 ]
  [   -1.000000     9.000000    -4.000000 ]
  [   -1.000000     8.000000    -3.000000 ]

Проверка Q^T·Q =
  [    1.000000     0.000000     0.000000 ]
  [    0.000000     1.000000     0.000000 ]
  [    0.000000     0.000000     1.000000 ]



## Собственные значения QR-алгоритмом

In [4]:
lams, k, Ak, hist = qr_eigenvalues(A, eps)

print(f"Итераций: {k}\n")
show(Ak, "A^(k)")
for i, lam in enumerate(lams, 1):
    print(f"lambda_{i} = {fmt(lam)}")

print(f"\nСумма lambda = {sum(lams).real:.6f},  след A = {sum(A[i][i] for i in range(len(A)))}")

Итераций: 20

A^(k) =
  [    4.854102     2.121316   -12.879978 ]
  [   -0.000000    -1.854100     7.912409 ]
  [    0.000000     0.000001     0.999998 ]

lambda_1 =     4.854102
lambda_2 =    -1.854100
lambda_3 =     0.999998

Сумма lambda = 4.000000,  след A = 4


Характеристический многочлен $\det(A - \lambda E) = -(\lambda^3 - 4\lambda^2 - 6\lambda + 9)
= -(\lambda - 1)(\lambda^2 - 3\lambda - 9)$, откуда точные значения:

$$\lambda = 1, \qquad \lambda = \frac{3 \pm 3\sqrt5}{2} \approx 4.854102,\ -1.854102.$$

In [5]:
exact = sorted([1.0, (3 + 3 * math.sqrt(5)) / 2, (3 - 3 * math.sqrt(5)) / 2])
got = sorted(l.real if isinstance(l, complex) else l for l in lams)
print("max |lambda - lambda_точн| =", f"{max(abs(a - b) for a, b in zip(got, exact)):.2e}")

max |lambda - lambda_точн| = 2.35e-06


Критерий останова ограничивает поддиагональные элементы, а не саму погрешность $\lambda$. Поэтому
фактическая ошибка может быть в несколько раз больше $\varepsilon$: здесь множитель порядка
$|a_{23}| / |\lambda_2 - \lambda_3| \approx 7.9 / 2.85 \approx 2.8$. Погрешность пропорциональна $\varepsilon$
и уменьшается вместе с ним:

In [6]:
print(f"{'eps':>8} | {'итераций':>8} | {'max |lambda - lambda_точн|':>26}")
print("-" * 49)
for p in range(2, 13, 2):
    e = 10.0 ** -p
    l_e, k_e, _, _ = qr_eigenvalues(A, e)
    g = sorted(complex(l).real for l in l_e)
    print(f"{e:8.0e} | {k_e:8} | {max(abs(a - b) for a, b in zip(g, exact)):26.2e}")

     eps | итераций | max |lambda - lambda_точн|
-------------------------------------------------
   1e-02 |        9 |                   2.09e-03
   1e-04 |       14 |                   9.56e-05
   1e-06 |       20 |                   2.35e-06
   1e-08 |       28 |                   1.69e-08
   1e-10 |       35 |                   2.24e-10
   1e-12 |       43 |                   1.60e-12


### Сходимость

Значения упорядочены по модулю: $|4.854| > |-1.854| > |1|$, а отношения
$|\lambda_2/\lambda_1| \approx 0.38$ и $|\lambda_3/\lambda_2| \approx 0.54$ задают скорость, с которой
обнуляются поддиагональные элементы.

In [7]:
print(f"{'k':>3} | " + " | ".join(f"{'столбец ' + str(j + 1):>11}" for j in range(len(A) - 1)))
print("-" * 32)
for i, s in enumerate(hist, 1):
    print(f"{i:3} | " + " | ".join(f"{v:11.3e}" for v in s[:-1]))

  k |   столбец 1 |   столбец 2
--------------------------------
  1 |   2.121e+00 |   3.077e-01
  2 |   4.243e+00 |   1.438e-02
  3 |   1.414e+00 |   2.562e-02
  4 |   7.071e-01 |   1.642e-02
  5 |   2.496e-01 |   8.918e-03
  6 |   9.867e-02 |   4.800e-03
  7 |   3.722e-02 |   2.605e-03
  8 |   1.428e-02 |   1.399e-03
  9 |   5.446e-03 |   7.566e-04
 10 |   2.082e-03 |   4.074e-04
 11 |   7.949e-04 |   2.199e-04
 12 |   3.037e-04 |   1.185e-04
 13 |   1.160e-04 |   6.396e-05
 14 |   4.430e-05 |   3.449e-05
 15 |   1.692e-05 |   1.860e-05
 16 |   6.464e-06 |   1.003e-05
 17 |   2.469e-06 |   5.412e-06
 18 |   9.431e-07 |   2.919e-06
 19 |   3.602e-07 |   1.574e-06
 20 |   1.376e-07 |   8.490e-07


## Пример с комплексными собственными значениями

QR-алгоритм должен работать для произвольных матриц. Проверим его на матрице, у которой есть
пара комплексно-сопряжённых собственных значений.

In [8]:
C = [
    [ 1,  3, 1],
    [-2,  1, 4],
    [ 1, -1, 2],
]
lams_C, k_C, Ck, _ = qr_eigenvalues(C, eps)
show(C, "C", prec=0)
print(f"Итераций: {k_C}\n")
show(Ck, "C^(k)")
for i, lam in enumerate(lams_C, 1):
    print(f"lambda_{i} = {fmt(lam)}")

C =
  [     1      3      1 ]
  [    -2      1      4 ]
  [     1     -1      2 ]

Итераций: 112

C^(k) =
  [    0.397707     3.154439    -1.240559 ]
  [   -3.323258     0.724205    -2.549598 ]
  [   -0.000001     0.000000     2.878088 ]

lambda_1 =     0.560956 + 3.233630i
lambda_2 =     0.560956 - 3.233630i
lambda_3 =     2.878088


## Проверка с помощью NumPy (необязательно)

In [9]:
try:
    import numpy as np
except ImportError:
    np = None
    print("NumPy не установлен — пропускаем сверку")


def compare_with_numpy(name, M, ours):
    # упорядочиваем оба набора одинаково, чтобы сравнивать значения попарно
    order = lambda z: (z.real, z.imag)
    ours = sorted(map(complex, ours), key=order)
    ref = sorted(map(complex, np.linalg.eigvals(np.array(M, float))), key=order)

    print(f"Матрица {name}:")
    print(f"  {'QR-алгоритм':>26} | {'numpy':>26}")
    for a, b in zip(ours, ref):
        print(f"  {fmt(a):>26} | {fmt(b):>26}")
    print(f"  max |lambda - lambda_numpy| = {max(abs(a - b) for a, b in zip(ours, ref)):.2e}\n")


if np is not None:
    compare_with_numpy("A", A, lams)
    compare_with_numpy("C", C, lams_C)

Матрица A:
                 QR-алгоритм |                      numpy
                   -1.854100 |                  -1.854102
                    0.999998 |                   1.000000
                    4.854102 |                   4.854102
  max |lambda - lambda_numpy| = 2.35e-06

Матрица C:
                 QR-алгоритм |                      numpy
        0.560956 - 3.233630i |       0.560956 - 3.233631i
        0.560956 + 3.233630i |       0.560956 + 3.233631i
                    2.878088 |                   2.878088
  max |lambda - lambda_numpy| = 5.33e-07



## Вывод

Реализовано QR-разложение методом отражений Хаусхолдера (проверено: $QR = A$, $Q^TQ = E$) и на его основе
QR-алгоритм для полной проблемы собственных значений. Алгоритм находит как вещественные значения
(по обнулению поддиагональной части столбца), так и комплексно-сопряжённые пары
(по стабилизации корней блока $2\times2$).

Для матрицы варианта все собственные значения вещественные: $\lambda_1 \approx 4.854102$,
$\lambda_2 \approx -1.854102$, $\lambda_3 = 1$. Они совпадают с аналитическим решением.
Поддиагональные элементы убывают геометрически со знаменателями
$|\lambda_2/\lambda_1| \approx 0.38$ и $|\lambda_3/\lambda_2| \approx 0.54$. Медленнее сходится второй столбец,
и именно он определяет число итераций.